# Hands_on_RX — GNU Radio receiver converted to Notebook
This notebook contains the code from `Hands_on_RX.py` organized into cells with explanations and usage notes.

Notes: The flowgraph uses `osmosdr`, `gnuradio`, and a Qt GUI. Running the GUI inside a notebook requires the `%gui qt` magic and a working X display. For many cases it's easier to run the original script from a terminal.

## Requirements
- Python environment with GNU Radio (3.10+), `osmosdr` drivers (e.g., HackRF), and `PyQt5`.
- If using the notebook UI: run `%gui qt` before starting the flowgraph.

If you don't have these installed in the notebook kernel, run the original script `Hands_on_RX.py` from a terminal inside the same environment.

In [1]:
# Imports from the original script (headless - no GUI)
from gnuradio import analog, blocks, digital, fft, gr
from gnuradio.fft import window
from gnuradio.filter import firdes
import sys
import signal
from argparse import ArgumentParser
from gnuradio.eng_arg import eng_float, intx
from gnuradio import eng_notation
import osmosdr
import time
import threading

# NOTE: Running this cell will only import libraries. Instantiate and run the flowgraph in a later cell.
# This notebook is headless (no PyQt5 / qtgui). GUI sinks were replaced by vector sinks.


## Installation guidance
GNU Radio and `osmosdr` drivers are often installed via your system package manager or conda.
If you must use pip, only some components are pip-installable; full GNU Radio (with GUI and SDR drivers) is best installed via apt or conda.

Recommended options:
- Debian/Ubuntu (apt): `sudo apt install gnuradio gr-osmosdr hackrf python3-pyqt5`
- Conda (conda-forge): `conda create -n gnuradio -c conda-forge gnuradio gr-osmosdr pyqt sip`
- Pip (only for some pure-Python deps): `pip install PyQt5 sip` — GNU Radio itself may not install correctly via pip.

Choose the method that matches your OS and privileges. If you want, run the commands below in a terminal; I can also add a `requirements.txt` or a helper script.

In [2]:
# Example shell commands (do NOT run inside this Python cell unless using a shell magic)
# Debian/Ubuntu (run in terminal):
# sudo apt update && sudo apt install -y gnuradio gr-osmosdr hackrf python3-pyqt5
#
# Conda (recommended if you cannot use apt):
# conda create -n gnuradio -c conda-forge gnuradio gr-osmosdr pyqt sip -y
# conda activate gnuradio
#
# Pip for small deps (inside virtualenv):
# pip install PyQt5 sip
print("See the comments in this cell for suggested install commands.")

See the comments in this cell for suggested install commands.


## Parameters
Set the main parameters used by the flowgraph. Adjust as needed before instantiating the class.

In [3]:
# Default parameters (match the original script)
carrier_freq = 435e6  # center frequency in Hz
gain = 70             # SDR gain setting
samp_rate = 2_000_000 # sample rate in samples/sec

# You can modify these values before running the flowgraph.

## Flowgraph class
Below is the `Hands_on_RX` class taken from `Hands_on_RX.py` with minimal changes — conserved structure and comments.
Run the following cell to define the class in the notebook kernel.

In [4]:
class Hands_on_RX(gr.top_block):
    def __init__(self, carrier_freq=435e6, gain=70, samp_rate=2000000):
        gr.top_block.__init__(self, "Hands_on_RX", catch_exceptions=True)
        # Headless: no Qt widgets or GUI elements
        self.flowgraph_started = threading.Event()

        ##################################################
        # Parameters
        ##################################################
        self.carrier_freq = carrier_freq
        self.gain = gain
        self.samp_rate = samp_rate

        ##################################################
        # Variables
        ##################################################
        self.pilot_symbols = pilot_symbols = ((1, 1, 1, -1,),)
        self.pilot_carriers = pilot_carriers = ((-21, -7, 7, 21,),)
        self.payload_mod = payload_mod = digital.constellation_qpsk()
        self.packet_length_tag_key = packet_length_tag_key = "packet_len"
        self.occupied_carriers = occupied_carriers = (list(range(-26, -21)) + list(range(-20, -7)) + list(range(-6, 0)) + list(range(1, 7)) + list(range(8, 21)) + list(range(22, 27)),)
        self.length_tag_key_rx = length_tag_key_rx = "frame_len"
        self.header_mod = header_mod = digital.constellation_bpsk()
        self.fft_len = fft_len = 64
        self.sync_word2 = sync_word2 = [0, 0, 0, 0, 0, 0, -1, -1, -1, -1, 1, 1, -1, -1, -1, 1, -1, 1, 1, 1, 1, 1, -1, -1, -1, -1, -1, 1, -1, -1, 1, -1, 0, 1, -1, 1, 1, 1, -1, 1, 1, 1, -1, 1, 1, 1, 1, -1, 1, -1, -1, -1, 1, -1, 1, -1, -1, -1, -1, 0, 0, 0, 0, 0]
        self.sync_word1 = sync_word1 = [0., 0., 0., 0., 0., 0., 0., 1.41421356, 0., -1.41421356, 0., 1.41421356, 0., -1.41421356, 0., -1.41421356, 0., -1.41421356, 0., 1.41421356, 0., -1.41421356, 0., 1.41421356, 0., -1.41421356, 0., -1.41421356, 0., -1.41421356, 0., -1.41421356, 0., 1.41421356, 0., -1.41421356, 0., 1.41421356, 0., 1.41421356, 0., 1.41421356, 0., -1.41421356, 0., 1.41421356, 0., 1.41421356, 0., 1.41421356, 0., -1.41421356, 0., 1.41421356, 0., 1.41421356, 0., 1.41421356, 0., 0., 0., 0., 0., 0.]
        self.rolloff = rolloff = 0
        self.payload_equalizer = payload_equalizer = digital.ofdm_equalizer_simpledfe(fft_len, payload_mod.base(), occupied_carriers, pilot_carriers, pilot_symbols, 1)
        self.packet_len = packet_len = 30
        self.length_tag_key = length_tag_key = "packet_len"
        self.header_len = header_len = 10
        self.header_formatter = header_formatter = digital.packet_header_ofdm(occupied_carriers, n_syms=1, len_tag_key=packet_length_tag_key, frame_len_tag_key=length_tag_key_rx, bits_per_header_sym=header_mod.bits_per_symbol(), bits_per_payload_sym=payload_mod.bits_per_symbol(), scramble_header=False)
        self.header_equalizer = header_equalizer = digital.ofdm_equalizer_simpledfe(fft_len, header_mod.base(), occupied_carriers, pilot_carriers, pilot_symbols)

        ##################################################
        # Blocks (headless adaptations)
        ##################################################
        # Vector sink to collect payload complex samples for inspection in the notebook
        self.payload_sink = blocks.vector_sink_c()

        self.osmosdr_source_0 = osmosdr.source(
            args="numchan=" + str(1) + " " + 'hackrf=0'
        )
        self.osmosdr_source_0.set_time_now(osmosdr.time_spec_t(time.time()), osmosdr.ALL_MBOARDS)
        self.osmosdr_source_0.set_sample_rate(samp_rate)
        self.osmosdr_source_0.set_center_freq(435e6, 0)
        self.osmosdr_source_0.set_freq_corr(0, 0)
        self.osmosdr_source_0.set_dc_offset_mode(2, 0)
        self.osmosdr_source_0.set_iq_balance_mode(2, 0)
        self.osmosdr_source_0.set_gain_mode(True, 0)
        self.osmosdr_source_0.set_gain(70, 0)
        self.osmosdr_source_0.set_if_gain(70, 0)
        self.osmosdr_source_0.set_bb_gain(70, 0)
        self.osmosdr_source_0.set_antenna('', 0)
        self.osmosdr_source_0.set_bandwidth(2e6, 0)
        self.fft_vxx_1 = fft.fft_vcc(fft_len, True, (), True, 1)
        self.fft_vxx_0_0 = fft.fft_vcc(fft_len, True, (), True, 1)
        self.digital_packet_headerparser_b_0 = digital.packet_headerparser_b(header_formatter.base())
        self.digital_ofdm_sync_sc_cfb_0 = digital.ofdm_sync_sc_cfb(fft_len, 16, False, 0.9)
        self.digital_ofdm_serializer_vcc_payload = digital.ofdm_serializer_vcc(fft_len, occupied_carriers, length_tag_key_rx, packet_length_tag_key, 1, '', True)
        self.digital_ofdm_serializer_vcc_header = digital.ofdm_serializer_vcc(fft_len, occupied_carriers, length_tag_key_rx, '', 0, '', True)
        self.digital_ofdm_frame_equalizer_vcvc_1 = digital.ofdm_frame_equalizer_vcvc(payload_equalizer.base(), 16, length_tag_key_rx, True, 0)
        self.digital_ofdm_frame_equalizer_vcvc_0 = digital.ofdm_frame_equalizer_vcvc(header_equalizer.base(), 16, length_tag_key_rx, True, 1)
        self.digital_ofdm_chanest_vcvc_0 = digital.ofdm_chanest_vcvc(sync_word1, sync_word2, 1, 0, 3, False)
        self.digital_header_payload_demux_0 = digital.header_payload_demux(
            3,
            fft_len,
            16,
            length_tag_key_rx,
            "",
            True,
            gr.sizeof_gr_complex,
            "rx_time",
            samp_rate,
            (),
            0)
        self.digital_crc32_bb_0_0 = digital.crc32_bb(True, packet_length_tag_key, True)
        self.digital_constellation_decoder_cb_1 = digital.constellation_decoder_cb(payload_mod.base())
        self.digital_constellation_decoder_cb_0 = digital.constellation_decoder_cb(header_mod.base())
        self.blocks_repack_bits_bb_0_1 = blocks.repack_bits_bb(payload_mod.bits_per_symbol(), 8, packet_length_tag_key, True, gr.GR_LSB_FIRST)
        self.blocks_multiply_xx_0 = blocks.multiply_vcc(1)
        self.blocks_file_sink_0 = blocks.file_sink(gr.sizeof_char*1, 'rx.txt', True)
        self.blocks_file_sink_0.set_unbuffered(False)
        self.blocks_delay_0 = blocks.delay(gr.sizeof_gr_complex*1, 80)
        self.analog_frequency_modulator_fc_0 = analog.frequency_modulator_fc((-2.0/fft_len))


        ##################################################
        # Connections
        ##################################################
        self.msg_connect((self.digital_packet_headerparser_b_0, 'header_data'), (self.digital_header_payload_demux_0, 'header_data'))
        self.connect((self.analog_frequency_modulator_fc_0, 0), (self.blocks_multiply_xx_0, 0))
        self.connect((self.blocks_delay_0, 0), (self.blocks_multiply_xx_0, 1))
        self.connect((self.blocks_multiply_xx_0, 0), (self.digital_header_payload_demux_0, 0))
        self.connect((self.blocks_repack_bits_bb_0_1, 0), (self.digital_crc32_bb_0_0, 0))
        self.connect((self.digital_constellation_decoder_cb_0, 0), (self.digital_packet_headerparser_b_0, 0))
        self.connect((self.digital_constellation_decoder_cb_1, 0), (self.blocks_repack_bits_bb_0_1, 0))
        self.connect((self.digital_crc32_bb_0_0, 0), (self.blocks_file_sink_0, 0))
        self.connect((self.digital_header_payload_demux_0, 0), (self.fft_vxx_0_0, 0))
        self.connect((self.digital_header_payload_demux_0, 1), (self.fft_vxx_1, 0))
        self.connect((self.digital_ofdm_chanest_vcvc_0, 0), (self.digital_ofdm_frame_equalizer_vcvc_0, 0))
        self.connect((self.digital_ofdm_frame_equalizer_vcvc_0, 0), (self.digital_ofdm_serializer_vcc_header, 0))
        self.connect((self.digital_ofdm_frame_equalizer_vcvc_1, 0), (self.digital_ofdm_serializer_vcc_payload, 0))
        self.connect((self.digital_ofdm_serializer_vcc_header, 0), (self.digital_constellation_decoder_cb_0, 0))
        self.connect((self.digital_ofdm_serializer_vcc_payload, 0), (self.digital_constellation_decoder_cb_1, 0))
        self.connect((self.digital_ofdm_serializer_vcc_payload, 0), (self.payload_sink, 0))
        self.connect((self.digital_ofdm_sync_sc_cfb_0, 0), (self.analog_frequency_modulator_fc_0, 0))
        self.connect((self.digital_ofdm_sync_sc_cfb_0, 1), (self.digital_header_payload_demux_0, 1))
        self.connect((self.fft_vxx_0_0, 0), (self.digital_ofdm_chanest_vcvc_0, 0))
        self.connect((self.fft_vxx_1, 0), (self.digital_ofdm_frame_equalizer_vcvc_1, 0))
        self.connect((self.osmosdr_source_0, 0), (self.blocks_delay_0, 0))
        self.connect((self.osmosdr_source_0, 0), (self.digital_ofdm_sync_sc_cfb_0, 0))

    # No Qt closeEvent in headless mode

    def get_carrier_freq(self):
        return self.carrier_freq

    def set_carrier_freq(self, carrier_freq):
        self.carrier_freq = carrier_freq

    def get_gain(self):
        return self.gain

    def set_gain(self, gain):
        self.gain = gain

    def get_samp_rate(self):
        return self.samp_rate

    def set_samp_rate(self, samp_rate):
        self.samp_rate = samp_rate
        self.osmosdr_source_0.set_sample_rate(self.samp_rate)

    def get_pilot_symbols(self):
        return self.pilot_symbols

    def set_pilot_symbols(self, pilot_symbols):
        self.pilot_symbols = pilot_symbols
        self.set_header_equalizer(digital.ofdm_equalizer_simpledfe(self.fft_len, header_mod.base(), self.occupied_carriers, self.pilot_carriers, self.pilot_symbols))
        self.set_payload_equalizer(digital.ofdm_equalizer_simpledfe(self.fft_len, payload_mod.base(), self.occupied_carriers, self.pilot_carriers, self.pilot_symbols, 1))

    def get_pilot_carriers(self):
        return self.pilot_carriers

    def set_pilot_carriers(self, pilot_carriers):
        self.pilot_carriers = pilot_carriers
        self.set_header_equalizer(digital.ofdm_equalizer_simpledfe(self.fft_len, header_mod.base(), self.occupied_carriers, self.pilot_carriers, self.pilot_symbols))
        self.set_payload_equalizer(digital.ofdm_equalizer_simpledfe(self.fft_len, payload_mod.base(), self.occupied_carriers, self.pilot_carriers, self.pilot_symbols, 1))

    def get_payload_mod(self):
        return self.payload_mod

    def set_payload_mod(self, payload_mod):
        self.payload_mod = payload_mod

    def get_packet_length_tag_key(self):
        return self.packet_length_tag_key

    def set_packet_length_tag_key(self, packet_length_tag_key):
        self.packet_length_tag_key = packet_length_tag_key
        self.set_header_formatter(digital.packet_header_ofdm(self.occupied_carriers, n_syms=1, len_tag_key=self.packet_length_tag_key, frame_len_tag_key=self.length_tag_key_rx, bits_per_header_sym=header_mod.bits_per_symbol(), bits_per_payload_sym=payload_mod.bits_per_symbol(), scramble_header=False))

    def get_occupied_carriers(self):
        return self.occupied_carriers

    def set_occupied_carriers(self, occupied_carriers):
        self.occupied_carriers = occupied_carriers
        self.set_header_equalizer(digital.ofdm_equalizer_simpledfe(self.fft_len, header_mod.base(), self.occupied_carriers, self.pilot_carriers, self.pilot_symbols))
        self.set_header_formatter(digital.packet_header_ofdm(self.occupied_carriers, n_syms=1, len_tag_key=self.packet_length_tag_key, frame_len_tag_key=self.length_tag_key_rx, bits_per_header_sym=header_mod.bits_per_symbol(), bits_per_payload_sym=payload_mod.bits_per_symbol(), scramble_header=False))
        self.set_payload_equalizer(digital.ofdm_equalizer_simpledfe(self.fft_len, payload_mod.base(), self.occupied_carriers, self.pilot_carriers, self.pilot_symbols, 1))

    def get_length_tag_key_rx(self):
        return self.length_tag_key_rx

    def set_length_tag_key_rx(self, length_tag_key_rx):
        self.length_tag_key_rx = length_tag_key_rx
        self.set_header_formatter(digital.packet_header_ofdm(self.occupied_carriers, n_syms=1, len_tag_key=self.packet_length_tag_key, frame_len_tag_key=self.length_tag_key_rx, bits_per_header_sym=header_mod.bits_per_symbol(), bits_per_payload_sym=payload_mod.bits_per_symbol(), scramble_header=False))

    def get_header_mod(self):
        return self.header_mod

    def set_header_mod(self, header_mod):
        self.header_mod = header_mod

    def get_fft_len(self):
        return self.fft_len

    def set_fft_len(self, fft_len):
        self.fft_len = fft_len
        self.set_header_equalizer(digital.ofdm_equalizer_simpledfe(self.fft_len, header_mod.base(), self.occupied_carriers, self.pilot_carriers, self.pilot_symbols))
        self.set_payload_equalizer(digital.ofdm_equalizer_simpledfe(self.fft_len, payload_mod.base(), self.occupied_carriers, self.pilot_carriers, self.pilot_symbols, 1))
        self.analog_frequency_modulator_fc_0.set_sensitivity((-2.0/self.fft_len))

    def get_sync_word2(self):
        return self.sync_word2

    def set_sync_word2(self, sync_word2):
        self.sync_word2 = sync_word2

    def get_sync_word1(self):
        return self.sync_word1

    def set_sync_word1(self, sync_word1):
        self.sync_word1 = sync_word1

    def get_rolloff(self):
        return self.rolloff

    def set_rolloff(self, rolloff):
        self.rolloff = rolloff

    def get_payload_equalizer(self):
        return self.payload_equalizer

    def set_payload_equalizer(self, payload_equalizer):
        self.payload_equalizer = payload_equalizer

    def get_packet_len(self):
        return self.packet_len

    def set_packet_len(self, packet_len):
        self.packet_len = packet_len

    def get_length_tag_key(self):
        return self.length_tag_key

    def set_length_tag_key(self, length_tag_key):
        self.length_tag_key = length_tag_key

    def get_header_len(self):
        return self.header_len

    def set_header_len(self, header_len):
        self.header_len = header_len

    def get_header_formatter(self):
        return self.header_formatter

    def set_header_formatter(self, header_formatter):
        self.header_formatter = header_formatter

    def get_header_equalizer(self):
        return self.header_equalizer

    def set_header_equalizer(self, header_equalizer):
        self.header_equalizer = header_equalizer

# End of class definition


In [5]:
def argument_parser():
    parser = ArgumentParser()
    parser.add_argument(
        "-f", "--carrier-freq", dest="carrier_freq", type=eng_float, default=eng_notation.num_to_str(float(435e6)),
        help="Set carrier_freq [default=%(default)r]")
    parser.add_argument(
        "-g", "--gain", dest="gain", type=eng_float, default=eng_notation.num_to_str(float(70)),
        help="Set gain [default=%(default)r]")
    parser.add_argument(
        "-s", "--samp-rate", dest="samp_rate", type=eng_float, default=eng_notation.num_to_str(float(2000000)),
        help="Set samp_rate [default=%(default)r]")
    parser.add_argument(
        "-d", "--duration", dest="duration", type=float, default=10.0,
        help="Run duration in seconds for headless execution [default=10s]")
    return parser


def main(top_block_cls=Hands_on_RX, options=None):
    if options is None:
        options = argument_parser().parse_args()

    # Headless main: start the top block, run for a bounded duration, then stop
    tb = top_block_cls(carrier_freq=options.carrier_freq, gain=options.gain, samp_rate=options.samp_rate)

    def sig_handler(sig=None, frame=None):
        try:
            tb.stop()
            tb.wait()
        except Exception:
            pass

    signal.signal(signal.SIGINT, sig_handler)
    signal.signal(signal.SIGTERM, sig_handler)

    tb.start()
    tb.flowgraph_started.set()

    print(f"Flowgraph started headless for {options.duration} seconds...")
    try:
        time.sleep(options.duration)
    except KeyboardInterrupt:
        pass

    tb.stop()
    tb.wait()
    print("Flowgraph stopped.")

# If running the notebook and you want to call main(), call it from the notebook or a terminal.


## How to run the flowgraph from this notebook
Two main options:

- **Inside the notebook GUI (experimental):** run a Qt event loop and then start the flowgraph. In a Jupyter environment use `%gui qt` and then run the cells below.
- **Recommended:** run `python3 Hands_on_RX.py` from the same environment/virtualenv (it will start the full GUI reliably).

In [6]:
# Example: start the flowgraph in the notebook (headless)
# Instantiate, start, wait a short time, then stop and wait.
# This prevents the kernel from blocking indefinitely.
#
# Example usage:
# tb = Hands_on_RX(carrier_freq=carrier_freq, gain=gain, samp_rate=samp_rate)
# tb.start()
# tb.flowgraph_started.set()
# time.sleep(10)  # or desired runtime
# tb.stop()
# tb.wait()
#
# After stopping you can inspect payload samples collected by the vector sink:
# samples = tb.payload_sink.data()

print("Notebook prepared for headless execution. Use the example above to run the flowgraph.")


Notebook prepared for headless execution. Use the example above to run the flowgraph.


In [7]:
tb = Hands_on_RX(carrier_freq=carrier_freq, gain=gain, samp_rate=samp_rate)
tb.start()
tb.flowgraph_started.set()
time.sleep(10)  # or desired runtime
tb.stop()
tb.wait()

gr-osmosdr 0.2.0.0 (0.2.0) gnuradio 3.10.11.0
built-in source types: file fcd rtl rtl_tcp uhd hackrf rfspace airspy soapy redpitaya 


RuntimeError: Failed to use '0' as HackRF device index: not enough devices

---
Saved notebook file: [RX_notebook.ipynb](RX_notebook.ipynb)
If you want, I can: run the notebook cells that define the class, or modify the notebook to include the omitted getters/setters.